# Instalación y configuración

## Paso 1: Descargar Docker Desktop

Ir a: [Link de descarga](https://www.docker.com/products/docker-desktop/)

En Windows:

- Ejecutar el instalador
- Aceptar configuración por defecto
- Reiniciar el computador si lo pide

```{note}
**Recordatorio:** Recuerda manetener Docker Desktop abierto mientras estemos trabajando con él

## Paso 2: Crear una carpeta para nuestro proyecto

Es necesario tener una carpeta en donde iremos dejando nuestros archivos para que todo funciones bien y este bien estructurado

## Paso 3: Buscar nuestros datos

En este caso lo haremos con un archivo CSV, recomiendo [esta página](https://datos.gob.cl/), para buscar distintos tipos de datos de interes. En este caso ocuparemos [estos](https://datos.gob.cl/dataset/precios-consumidor1), los cuales son los precios al consumidor de los principales alimentos de la canasta familiar que son capturados en supermercados, ferias libres, carnicerías y panaderías del país. 

```{note}
**Recordatorio:** Dejarlos dentro de nuestra carpeta



## Paso 4: Configurar Logstash

Para esto haremos un pipeline de Logstash el cual tiene siempre 3 partes:

1. input: ¿De dónde vienen los datos?
2. filter: ¿Cómo transformamos esos datos?
3. output: ¿A dónde enviamos esos datos?

Sigue exactamente esta estructura en este caso

![Esquema](imagenes/Logstash.png)

### Sección input
```
input {
  file {
    path => "/datos_csv/precio_consumidor_publico_2025.csv"
    start_position => "beginning"
    sincedb_path => "/dev/null"
    codec => plain { charset => "UTF-8" }
  }
}
```
**path**  →  es la ruta de nuestro archivo

**start_position**  →  para que lea desde el inicio por eso beginning

**sincedb_path**  →  para que no recuerde nada, cada vez que inicie vuelve a leer todo, en caso contrario eliminar esta linea

**codec**  →  indica como interpretar los caracteres

### Sección filter
```
filter {
  csv {
    separator => ","
    quote_char => '"'
    skip_header => true
    columns => [
      "anio","mes","semana","fecha_inicio","fecha_termino","id_region",
      "region","sector","tipo_punto","grupo","producto","unidad",
      "precio_minimo","precio_maximo","precio_promedio"
    ]
  }
  mutate {
    gsub => [
      "precio_promedio", ",", "."
    ]
  }
  mutate {
    convert => {
      "anio" => "integer"
      "mes" => "integer"
      "semana" => "integer"
      "id_region" => "integer"
      "precio_minimo" => "float"
      "precio_maximo" => "float"
      "precio_promedio" => "float"
    }
  }
  date {
    match => [ "fecha_inicio", "yyyy-MM-dd" ]
    target => "@timestamp"
  }
}
```

**separator** → define que nuestro csv esta separado por comas

**quote_char** → indica que los textos estan entre comillas

**skip_header** → No procesamos la primera fila del csv

**columns** → Aqui definimos manualmente el nombre de cada columna

**mutate** → esta opción nos sirve para reemplazar por ejemplo comas por puntos, transformar los tipos de datos, definimos un formato que elasticsearch pueda leer como por ejemplo en la fecha.
### Sección output
```
output {
  elasticsearch {
    hosts => ["http://elasticsearch:9200"]
    index => "precios_consumidor_2026"
    manage_template => false
  }
}
```
**hosts** → la dirección del servidor donde logstash va a enviar los datos

**index** → nuestro indice donde se guardaran los datos

**manage_template**  → ya que convertimos los tipos correctamente, no es necesario que elasticsearch haga una plantilla o configuraciones por defecto, por eso false.


```{note}
**Recordatorio:** Todo el codigo de abajo tiene que estar dentro de un archivo pipeline.conf y a su vez en nuestra carpeta

In [ ]:
input {
  file {
    path => "/datos_csv/precio_consumidor_publico_2025.csv"
    start_position => "beginning"
    sincedb_path => "/dev/null"
    codec => plain { charset => "UTF-8" }
  }
}

filter {
  csv {
    separator => ","
    quote_char => '"'
    skip_header => true
    columns => [
      "anio","mes","semana","fecha_inicio","fecha_termino","id_region",
      "region","sector","tipo_punto","grupo","producto","unidad",
      "precio_minimo","precio_maximo","precio_promedio"
    ]
  }
  mutate {
    gsub => [
      "precio_promedio", ",", "."
    ]
  }
  mutate {
    convert => {
      "anio" => "integer"
      "mes" => "integer"
      "semana" => "integer"
      "id_region" => "integer"
      "precio_minimo" => "float"
      "precio_maximo" => "float"
      "precio_promedio" => "float"
    }
  }
  date {
    match => [ "fecha_inicio", "yyyy-MM-dd" ]
    target => "@timestamp"
  }
}

output {
  elasticsearch {
    hosts => ["http://elasticsearch:9200"]
    index => "precios_consumidor_2025"
    manage_template => false
  }
}

## Paso 5: Crear nuestro docker-compose.yml


In [ ]:
services:
  elasticsearch:
    container_name: elasticsearch
    image: docker.elastic.co/elasticsearch/elasticsearch:8.11.1
    environment:
      - discovery.type=single-node
      - xpack.security.enabled=false
      - ES_JAVA_OPTS=-Xms512m -Xmx512m
    ports:
      - 9200:9200
    networks:
      - elastic
    logging:
      driver: "json-file"
      options:
        max-size: "10m"
        max-file: "3"

  logstash:
    container_name: logstash
    image: docker.elastic.co/logstash/logstash:8.11.1
    volumes:
      - ./pipeline.conf:/usr/share/logstash/pipeline/logstash.conf:ro
      - ./:/datos_csv/:ro  
    environment:
      - LS_JAVA_OPTS=-Xms256m -Xmx256m
    networks:
      - elastic
    depends_on:
      - elasticsearch
    logging:
      driver: "json-file"
      options:
        max-size: "10m"
        max-file: "3"

  kibana:
    container_name: kibana
    image: docker.elastic.co/kibana/kibana:8.11.1
    ports:
      - 5601:5601
    networks:
      - elastic
    depends_on:
      - elasticsearch
    logging:
      driver: "json-file"
      options:
        max-size: "10m"
        max-file: "3"

networks:
  elastic:
    driver: bridge